## Thesis Chapter 3: Neural underpinnings of DD and brain-behavior questions

All additional plots & stats that are not already in main DD paper
e.g. correlations

### nPRF model stuff

- figure on null results of group comparison in main paper
-

In [3]:
import numpy as np
import pandas as pd
import os.path as op

BIDS_ROOT = '/Users/mrenke/data/ds-dnumrisk'
group_df = pd.read_csv(op.join(BIDS_ROOT, 'group_assignment.csv')).set_index('subject')

PHENOTYPE_DIR = BIDS_ROOT + '/derivatives/phenotype'

from numrisk.behavior_general.measures_registry import _load_magjudge_bauer, _load_magjudge_probit, _load_decode_r
magjudge_bauer_pcm = _load_magjudge_bauer(variant = 'v4_choice') # , suffix_columns=False
magjudge_bauer_pcm_RDM = _load_magjudge_bauer(variant = 'v4_rdm')
magjudge_probit = _load_magjudge_probit()

nPRF_r = _load_decode_r()

df_comb = nPRF_r.join(magjudge_bauer_pcm_RDM, how='inner').join(magjudge_bauer_pcm, how='inner').join(magjudge_probit, how='inner')


/Users/mrenke/mambaforge/envs/behav_fit/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [6]:

import pingouin

for behav_var in ['gamma_magjudge','perceptual_noise_sd_v4_choice','memory_noise_sd_v4_choice','perceptual_noise_sd_v4_rdm', 'memory_noise_sd_v4_rdm']: #  'perceptual_noise_sd_unbiased',
    cor = pingouin.corr(df_comb['neural_numsense_precision'], df_comb[behav_var], method='spearman')  # 'spearman' or 'shepard'
    r_, p_ = np.round(cor['r'].iloc[0], 2), np.round(cor['p-val'].iloc[0], 3)
    print(f'correlation neural_numsense_precision vs {behav_var}: r={r_}, p={p_}')



correlation neural_numsense_precision vs gamma_magjudge: r=0.14, p=0.264
correlation neural_numsense_precision vs perceptual_noise_sd_v4_choice: r=-0.13, p=0.302
correlation neural_numsense_precision vs memory_noise_sd_v4_choice: r=-0.05, p=0.71
correlation neural_numsense_precision vs perceptual_noise_sd_v4_rdm: r=-0.25, p=0.04
correlation neural_numsense_precision vs memory_noise_sd_v4_rdm: r=-0.04, p=0.727


## Connectivity

### Gradient stuff:
newest in notebook: `parietal_patterns/gradients_noHalo/rep_groupDiffs_dParams.ipynb`

### PFM

In [7]:
from numrisk.behavior_general.measures_registry import _load_npc_dispersion #_load_npc_pfm_net_area # only has DAN & visual2

npc_pfm_net_area = pd.read_csv(op.join(PHENOTYPE_DIR, 'netsPFM_NPC_allNets_atlas-caNets_DDnr_method-individual_area.csv'))
npc_pfm_net_area = npc_pfm_net_area.set_index(['subject', 'network']).unstack('network')
npc_pfm_net_area.columns = [f'{col[1]}_{col[0]}' for col in npc_pfm_net_area.columns]

npc_grad_dispersion = _load_npc_dispersion()

df_comb = npc_pfm_net_area.join(npc_grad_dispersion, how='inner')

/Users/mrenke/mambaforge/envs/behav_fit/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [8]:
print(f'Correlation NPC_dispersion vs: \n')
networks = npc_pfm_net_area.columns

for net_var in networks:
    cor = pingouin.corr(df_comb['NPC_dispersion'], df_comb[net_var], method='spearman')  # 'spearman' or 'shepard'
    r_, p_ = np.round(cor['r'].iloc[0], 2), np.round(cor['p-val'].iloc[0], 3)
    N_valid_datapoints = df_comb[['NPC_dispersion', net_var]].dropna().shape[0]
    print(f'{net_var}: r={r_}, p={p_}, N={N_valid_datapoints}')


Correlation NPC_dispersion vs: 

Auditory_size: r=-1.0, p=0.0, N=3
Cingulo-Opercular_size: r=-0.14, p=0.302, N=58
Default_size: r=0.21, p=0.118, N=58
Dorsal-attention_size: r=-0.34, p=0.005, N=66
Frontoparietal_size: r=0.41, p=0.023, N=31
Somatomotor_size: r=0.04, p=0.759, N=66
Visual2_size: r=0.48, p=0.0, N=66


In [9]:
npc_pfm_net_area.mean(axis=0).sort_values(ascending=False).to_frame(name='mean_area').join(npc_pfm_net_area.std(axis=0).sort_values(ascending=False).to_frame(name='SD_area'))

,mean_area,SD_area
Dorsal-attention_size,67.615374,13.492762
Somatomotor_size,13.170410,7.421524
Visual2_size,10.326867,8.886962
Cingulo-Opercular_size,2.613277,2.974690
Frontoparietal_size,1.874685,3.463420
Default_size,1.827932,1.558278
Auditory_size,1.250184,0.733597


##### Network coverage within the NPC mask

Used in manuscript text (`projects/dnumrisk/rework_neural_section_13-07-26.tex`, second Results paragraph): only Visual2, Dorsal-attention, and Somatomotor are present in *every* subject's NPC mask -- the other four networks are only sparsely represented, so mean/SD and correlation numbers for those would be unreliable.

In [10]:
# N subjects per network within the NPC mask -- which networks are universally represented?
npc_pfm_net_area_raw = pd.read_csv(op.join(PHENOTYPE_DIR, 'netsPFM_NPC_allNets_atlas-caNets_DDnr_method-individual_area.csv'))
coverage = npc_pfm_net_area_raw.groupby('network')['subject'].nunique().sort_values(ascending=False)
print(f'Total subjects in file: {npc_pfm_net_area_raw["subject"].nunique()}')
print(coverage)

Total subjects in file: 66
network
Dorsal-attention     66
Somatomotor          66
Visual2              66
Cingulo-Opercular    58
Default              58
Frontoparietal       31
Auditory              3
Name: subject, dtype: int64


##### Group comparison NPC area

whole brain in: `/parietal_patterns/nets_PFM/network_size_analysis.ipynb`

In [11]:
## Group comparisons:
# Summary statistics table + uncorrected t-tests
from scipy import stats
npc_pfm_net_area = pd.read_csv(op.join(PHENOTYPE_DIR, 'netsPFM_NPC_allNets_atlas-caNets_DDnr_method-individual_area.csv'))
networks = npc_pfm_net_area['network'].unique()

rows = []
for net in networks:
    sub_data = npc_pfm_net_area[npc_pfm_net_area['network'] == net].set_index('subject').join(group_df, how='inner').reset_index()
    g0 = sub_data[sub_data['group'] == 0]['size'].values
    g1 = sub_data[sub_data['group'] == 1]['size'].values
    t, p = stats.ttest_ind(g0, g1) if (len(g0) > 1 and len(g1) > 1) else (np.nan, np.nan)
    rows.append({
        'network': net,
        'group0_mean': np.round(g0.mean(), 3),
        'group0_std':  np.round(g0.std(), 3),
        'group1_mean': np.round(g1.mean(), 3),
        'group1_std':  np.round(g1.std(), 3),
        't': np.round(t, 3), 'p_uncorrected':np.round(p, 4),
    })

stats_df = (pd.DataFrame(rows)
            .sort_values('p_uncorrected')
            .reset_index(drop=True))

# Bonferroni correction
n_tests = stats_df['p_uncorrected'].notna().sum()
stats_df['p_bonferroni'] = (stats_df['p_uncorrected'] * n_tests).clip(upper=1.0)

pd.set_option('display.float_format', '{:.3f}'.format)
display(stats_df)

,network,group0_mean,group0_std,group1_mean,group1_std,t,p_uncorrected,p_bonferroni
0,Visual2,6.718,6.234,13.936,9.521,-3.588,0.001,0.004
1,Dorsal-attention,72.293,12.764,62.938,12.325,2.983,0.004,0.024
2,Cingulo-Opercular,1.852,1.585,3.375,3.704,-2.000,0.050,0.302
3,Frontoparietal,1.153,1.281,2.396,4.261,-0.985,0.333,1.000
4,Default,1.780,1.588,1.879,1.495,-0.239,0.812,1.000
5,Somatomotor,12.952,7.687,13.388,7.022,-0.237,0.814,1.000
6,Auditory,1.462,0.636,0.827,0.000,NaN,NaN,NaN


Only Visual2, Dorsal-attention, and Somatomotor are reported since those are the only networks present in every subject's NPC mask (see coverage check above).

In [21]:
from parietal_patterns.utils.statistics import between_group_comparison

rows_mwu = []
for net in ['Visual2', 'Dorsal-attention', 'Somatomotor']:
    net_data = npc_pfm_net_area_raw[npc_pfm_net_area_raw['network'] == net].set_index('subject').join(group_df, how='inner').reset_index()
    g0 = net_data[net_data['group'] == 0]['size'].values
    g1 = net_data[net_data['group'] == 1]['size'].values
    stats, stats_term = between_group_comparison(net_data, 'size', group_names= [0, 1])
    rows_mwu.append({'network': net, 'N0': len(g0), 'N1': len(g1), 'stats': f'{stats_term}={stats.statistic}', 'p': stats.pvalue})

pd.DataFrame(rows_mwu)

,network,N0,N1,stats,p
0,Visual2,33,33,"U(33, 33)=295.0",0.001
1,Dorsal-attention,33,33,t(64)=2.9825253424755727,0.004
2,Somatomotor,33,33,t(64)=-0.2368428597768444,0.814


### DAN patches

In [6]:
nets_DANpatch_sizes = pd.read_csv(op.join(PHENOTYPE_DIR, 'netsPFM_DANpatches_refAtlas-caNets_DDnr.csv')).rename(columns={'sub_id': 'subject'})
nets_DANpatch_sizes.head()

,subject,patch,ind_area,n_verts
0,1,L_frontal-lateral,28.938550,902
1,1,L_frontal-medial-dorsal,4.083290,147
2,1,L_parietal-lateral,48.147021,2264
3,1,L_temporal,18.329310,523
4,1,R_frontal-lateral,45.890044,1201


In [7]:
var = 'ind_area'

rows = []
for patch in nets_DANpatch_sizes['patch'].unique():
    patch_data = nets_DANpatch_sizes[nets_DANpatch_sizes['patch'] == patch].set_index('subject').join(group_df, how='inner').reset_index()
    g0 = patch_data[patch_data['group'] == 0][var].values
    g1 = patch_data[patch_data['group'] == 1][var].values
    stats, stats_term = between_group_comparison(patch_data, var, group_names= [0, 1])
    rows.append({'patch': patch, 'N0': len(g0), 'N1': len(g1), 
                    'group0_mean': np.round(g0.mean(), 3),
                    'group0_std':  np.round(g0.std(), 3),
                    'group1_mean': np.round(g1.mean(), 3),
                    'group1_std':  np.round(g1.std(), 3),                     
                    'stats': f'{stats_term}={stats.statistic}', 
                    'p_uncorrected': stats.pvalue})

#pd.DataFrame(rows)


NameError: name 'between_group_comparison' is not defined

In [37]:
stats_df = (pd.DataFrame(rows)
            .sort_values('p_uncorrected')
            .reset_index(drop=True))

# Bonferroni correction
n_tests = stats_df['p_uncorrected'].notna().sum()
stats_df['p_bonferroni'] = (stats_df['p_uncorrected'] * n_tests).clip(upper=1.0)

pd.set_option('display.float_format', '{:.3f}'.format)
display(stats_df)

,patch,N0,N1,group0_mean,group0_std,group1_mean,group1_std,stats,p_uncorrected,p_bonferroni
0,L_frontal-lateral,33,32,36.015,14.461,25.041,16.165,"U(33, 32)=804.0",0.000,0.002
1,R_parietal-lateral,33,33,62.180,14.292,51.893,11.217,"U(33, 33)=769.0",0.004,0.033
2,L_parietal-lateral,33,33,55.823,13.398,49.054,11.306,"U(33, 33)=718.0",0.027,0.212
3,L_temporal,33,33,11.929,5.991,14.457,7.405,"U(33, 33)=443.0",0.195,1.000
4,R_frontal-lateral,31,32,41.912,21.425,36.101,21.699,t(61)=1.0521364162037292,0.297,1.000
5,L_frontal-medial-dorsal,33,33,7.443,5.688,9.165,6.985,"U(33, 33)=491.0",0.497,1.000
6,R_temporal,32,30,12.451,6.624,13.242,7.689,t(60)=-0.427642578894036,0.670,1.000
7,R_frontal-medial-dorsal,30,28,7.205,6.778,7.197,6.092,"U(30, 28)=409.0",0.870,1.000


### Comparing correlations against each other

For questions like "is the DAN patch that correlates with vs-IQ actually correlating
*more strongly* than another patch does" -- a plain eyeball comparison of two r/p pairs
isn't a test. Since these correlations share a variable and come from the same sample
(not two independent groups), the right tool is a test for two *dependent, overlapping*
correlations: Meng, Rosenthal & Rubin (1992, Psych.\ Bull.), the same test the R package
`cocor` implements as `cocor.dep.groups.overlap()`. Implemented here directly in Python
(no `cocor`/R dependency) since the formula is short; verified below by Monte Carlo
before trusting it on real data -- simulated data under the null (true r1 == r2) gives a
~5% false-positive rate at alpha=0.05 across several r/overlap combinations, and clear
power to detect a genuine difference, so the implementation itself checks out.

This covers: comparing two *neural* variables' correlation with the same behavioral
measure, or one neural variable's correlation with two different behavioral measures --
both share one variable, so both are the same "dependent/overlapping" case.

Not yet included: comparing the *same* correlation across the two independent groups
(control vs.\ DD) -- that needs a different, simpler test (Fisher r-to-z for independent
samples), left for when that specific question comes up.

In [4]:
from scipy.stats import norm, spearmanr

def meng1992_z(r_yx1, r_yx2, r_x1x2, n):
    """Meng, Rosenthal & Rubin (1992) test for two dependent, overlapping correlations
    sharing variable Y: compares r(Y, X1) vs r(Y, X2), given r(X1, X2), same sample of
    size n. Returns (z, p), two-tailed. Validated by Monte Carlo (see markdown above).
    """
    z1, z2 = np.arctanh(r_yx1), np.arctanh(r_yx2)
    rm2 = (r_yx1**2 + r_yx2**2) / 2
    f = min((1 - r_x1x2) / (2 * (1 - rm2)), 1.0)
    h = (1 - f * rm2) / (1 - rm2)
    z = (z1 - z2) * np.sqrt((n - 3) / (2 * (1 - r_x1x2) * h))
    p = 2 * (1 - norm.cdf(abs(z)))
    return z, p

#### Worked examples

Using visuospatial IQ (`me`, the "matrices"/fluid-reasoning IQ screener subscore --
same variable as "IQ screener (visuospatial reasoning)" in Table 1 of the manuscript)
and verbal IQ (`kn`) from `iq-scores_ids2.csv`, plus the DAN patch areas loaded above.

In [8]:
# Example 1: two NEURAL variables' correlation with the same behavioral measure --
# does L_frontal-lateral relate to vs-IQ more/less strongly than R_parietal-lateral does?
iq = pd.read_csv(op.join(PHENOTYPE_DIR, 'iq-scores_ids2.csv')).set_index('subject')
patch_wide = nets_DANpatch_sizes.set_index(['subject', 'patch'])['ind_area'].unstack('patch')
df_iq = patch_wide.join(iq, how='inner')

sub = df_iq[['L_frontal-lateral', 'R_parietal-lateral', 'me']].dropna()
n = len(sub)
r_y1, _ = spearmanr(sub['L_frontal-lateral'], sub['me'])
r_y2, _ = spearmanr(sub['R_parietal-lateral'], sub['me'])
r_12, _ = spearmanr(sub['L_frontal-lateral'], sub['R_parietal-lateral'])
z, p = meng1992_z(r_y1, r_y2, r_12, n)

print(f'N={n}')
print(f'r(L_frontal-lateral, vs-IQ)  = {r_y1:.3f}')
print(f'r(R_parietal-lateral, vs-IQ) = {r_y2:.3f}')
print(f'r(L_frontal-lateral, R_parietal-lateral) = {r_12:.3f}  (overlap between the two)')
print(f'Meng (1992) z={z:.3f}, p={p:.4f}')

N=65
r(L_frontal-lateral, vs-IQ)  = 0.211
r(R_parietal-lateral, vs-IQ) = 0.410
r(L_frontal-lateral, R_parietal-lateral) = 0.500  (overlap between the two)
Meng (1992) z=-1.668, p=0.0953


In [9]:
# Example 2: one NEURAL variable's correlation with two different BEHAVIORAL measures --
# does R_parietal-lateral relate to vs-IQ more/less strongly than to verbal IQ?
sub2 = df_iq[['R_parietal-lateral', 'me', 'kn']].dropna()
n2 = len(sub2)
r_y1b, _ = spearmanr(sub2['R_parietal-lateral'], sub2['me'])
r_y2b, _ = spearmanr(sub2['R_parietal-lateral'], sub2['kn'])
r_12b, _ = spearmanr(sub2['me'], sub2['kn'])
z2, p2 = meng1992_z(r_y1b, r_y2b, r_12b, n2)

print(f'N={n2}')
print(f'r(R_parietal-lateral, vs-IQ)     = {r_y1b:.3f}')
print(f'r(R_parietal-lateral, verbal-IQ) = {r_y2b:.3f}')
print(f'r(vs-IQ, verbal-IQ) = {r_12b:.3f}  (overlap between the two)')
print(f'Meng (1992) z={z2:.3f}, p={p2:.4f}')

N=66
r(R_parietal-lateral, vs-IQ)     = 0.419
r(R_parietal-lateral, verbal-IQ) = 0.270
r(vs-IQ, verbal-IQ) = 0.159  (overlap between the two)
Meng (1992) z=1.000, p=0.3173
